In [1]:
# Phase 2: MRMR Feature Selection + Model Training per Fold

# IMPORT LIBRARIES
import pandas as pd
import numpy as np
import os
from mrmr import mrmr_classif

from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# SETUP DATA
data_dir = "../data/"

folders = sorted([
    f for f in os.listdir(data_dir)
    if os.path.isdir(os.path.join(data_dir, f))
])

print("Folders found:", folders)
assert len(folders) == 16, "Expected 16 datasets!"


Folders found: ['1', '10', '11', '12', '13', '14', '15', '16', '2', '3', '4', '5', '6', '7', '8', '9']


In [3]:
# MODELS & PARAMS
models = {
    "SVM": (
        SVC(),
        {
            "C": [0.1, 1, 10],
            "kernel": ["rbf"],
            "gamma": ["scale", "auto"]
        }
    ),
    "KNN": (
        KNeighborsClassifier(),
        {
            "n_neighbors": [3, 5, 7, 9],
            "weights": ["uniform", "distance"]
        }
    ),
    "DT": (
        DecisionTreeClassifier(),
        {
            "max_depth": [None, 5, 10, 20],
            "min_samples_split": [2, 5, 10]
        }
    ),
    "RF": (
        RandomForestClassifier(),
        {
            "n_estimators": [100, 200],
            "max_depth": [None, 10, 20],
            "min_samples_split": [2, 5]
        }
    ),
    "MLP": (
        MLPClassifier(max_iter=500),
        {
            "hidden_layer_sizes": [(50,), (100,), (100, 50)],
            "learning_rate_init": [0.001, 0.01],
            "alpha": [0.0001, 0.001]
        }
    )
}
print("Models and hyperparameters defined.")

Models and hyperparameters defined.


In [4]:
# PHASE 2 RESULTS STORAGE
fold_results_list = []


# PROCESS EACH DATASET
for folder in folders:
    print(f"\nProcessing Dataset {folder}...")

    train_path = os.path.join(data_dir, folder, "train.csv")
    df = pd.read_csv(train_path)

    X = df.iloc[:, :-1]
    y = df.iloc[:, -1]

    # 10-FOLD CV
    kf = KFold(n_splits=10, shuffle=True, random_state=42)

    for fold_num, (train_idx, val_idx) in enumerate(kf.split(X), 1):
        print(f"\n  Fold {fold_num}")

        X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]


        # MISSING VALUE IMPUTATION
        for col in X_train.columns:
            mean_val = X_train[col].mean()
            X_train[col] = X_train[col].fillna(mean_val)
            X_val[col] = X_val[col].fillna(mean_val)

        # MRMR FEATURE SELECTION
        n_selected_features = min(20, X_train.shape[1])  # adjust if needed
        selected_features = mrmr_classif(X_train, y_train, K=n_selected_features)
        X_train_fs = X_train[selected_features]
        X_val_fs = X_val[selected_features]

        # SCALING
        scaler = StandardScaler()
        X_train_fs = scaler.fit_transform(X_train_fs)
        X_val_fs = scaler.transform(X_val_fs)


        # TRAIN MODELS
        fold_result = {"Fold": fold_num,
                       "No. Selected Features": len(selected_features),
                       "Features Name": ", ".join(selected_features)}

        for name, (model, params) in models.items():
            grid = GridSearchCV(model, params, cv=3, scoring='f1_macro', n_jobs=-1)
            grid.fit(X_train_fs, y_train)

            best_model = grid.best_estimator_
            y_pred = best_model.predict(X_val_fs)

            acc = accuracy_score(y_val, y_pred)
            f1 = f1_score(y_val, y_pred, average='macro')
            best_params = grid.best_params_

            # Save results in proper columns
            fold_result[f"{name}-Accuracy"] = f"{acc:.4f}"
            fold_result[f"{name}-F1"] = f"{f1:.4f}"
            fold_result[f"FS/DS Parameters ({name})"] = str(best_params)

            print(f"    {name} | Acc: {acc:.4f} | F1: {f1:.4f} | Params: {best_params}")

        fold_results_list.append(fold_result)



Processing Dataset 1...

  Fold 1


100%|██████████| 13/13 [00:02<00:00,  6.08it/s]


    SVM | Acc: 0.9677 | F1: 0.9511 | Params: {'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}
    KNN | Acc: 0.9771 | F1: 0.9647 | Params: {'n_neighbors': 3, 'weights': 'distance'}
    DT | Acc: 1.0000 | F1: 1.0000 | Params: {'max_depth': None, 'min_samples_split': 2}
    RF | Acc: 0.9990 | F1: 0.9987 | Params: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
    MLP | Acc: 0.9802 | F1: 0.9745 | Params: {'alpha': 0.001, 'hidden_layer_sizes': (100, 50), 'learning_rate_init': 0.001}

  Fold 2


100%|██████████| 13/13 [00:01<00:00,  7.37it/s]


    SVM | Acc: 0.9781 | F1: 0.7170 | Params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
    KNN | Acc: 0.9833 | F1: 0.7139 | Params: {'n_neighbors': 3, 'weights': 'distance'}
    DT | Acc: 1.0000 | F1: 1.0000 | Params: {'max_depth': None, 'min_samples_split': 2}
    RF | Acc: 1.0000 | F1: 1.0000 | Params: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
    MLP | Acc: 0.9937 | F1: 0.9835 | Params: {'alpha': 0.001, 'hidden_layer_sizes': (100, 50), 'learning_rate_init': 0.001}

  Fold 3


...

    SVM | Acc: 0.8836 | F1: 0.4022 | Params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
    KNN | Acc: 0.8860 | F1: 0.4123 | Params: {'n_neighbors': 9, 'weights': 'distance'}
    DT | Acc: 0.8836 | F1: 0.4000 | Params: {'max_depth': None, 'min_samples_split': 2}
    RF | Acc: 0.8836 | F1: 0.4000 | Params: {'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 200}
    MLP | Acc: 0.8836 | F1: 0.4000 | Params: {'alpha': 0.001, 'hidden_layer_sizes': (50,), 'learning_rate_init': 0.001}

  Fold 9


100%|██████████| 20/20 [00:02<00:00,  7.37it/s]


    SVM | Acc: 0.8920 | F1: 0.3873 | Params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
    KNN | Acc: 0.8872 | F1: 0.3122 | Params: {'n_neighbors': 9, 'weights': 'distance'}
    DT | Acc: 0.8920 | F1: 0.3876 | Params: {'max_depth': None, 'min_samples_split': 5}
    RF | Acc: 0.8920 | F1: 0.3891 | Params: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}
    MLP | Acc: 0.8872 | F1: 0.3137 | Params: {'alpha': 0.001, 'hidden_layer_sizes': (100, 50), 'learning_rate_init': 0.01}

  Fold 10


100%|██████████| 20/20 [00:02<00:00,  6.95it/s]


    SVM | Acc: 0.9028 | F1: 0.3693 | Params: {'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}
    KNN | Acc: 0.9004 | F1: 0.3674 | Params: {'n_neighbors': 9, 'weights': 'distance'}
    DT | Acc: 0.9016 | F1: 0.3613 | Params: {'max_depth': None, 'min_samples_split': 2}
    RF | Acc: 0.8992 | F1: 0.3572 | Params: {'max_depth': 20, 'min_samples_split': 5, 'n_estimators': 100}
    MLP | Acc: 0.9004 | F1: 0.3546 | Params: {'alpha': 0.001, 'hidden_layer_sizes': (50,), 'learning_rate_init': 0.001}


In [5]:
# CONVERT TO DATAFRAME
fold_results_df = pd.DataFrame(fold_results_list)

# Sort columns nicely
cols_order = ["Fold", "SVM-Accuracy", "SVM-F1", "KNN-Accuracy", "KNN-F1",
              "DT-Accuracy", "DT-F1", "RF-Accuracy", "RF-F1", "MLP-Accuracy", "MLP-F1",
              "No. Selected Features", "Features Name",
              "FS/DS Parameters (SVM)", "FS/DS Parameters (KNN)",
              "FS/DS Parameters (DT)", "FS/DS Parameters (RF)", "FS/DS Parameters (MLP)"]

fold_results_df = fold_results_df[cols_order]



In [6]:
# SAVE FOLD RESULTS CSV
fold_results_df.to_csv("phase2_fold_results.csv", index=False)
print("\nFold-wise results saved to 'phase2_fold_results.csv'")



Fold-wise results saved to 'phase2_fold_results.csv'


In [7]:

# AGGREGATE FINAL RESULTS (MEAN ± STD)
final_results = []

for name in ["SVM", "KNN", "DT", "RF", "MLP"]:
    acc_mean = fold_results_df[f"{name}-Accuracy"].astype(float).mean()
    acc_std = fold_results_df[f"{name}-Accuracy"].astype(float).std()

    f1_mean = fold_results_df[f"{name}-F1"].astype(float).mean()
    f1_std = fold_results_df[f"{name}-F1"].astype(float).std()

    # Most frequent best params
    best_params = fold_results_df[f"FS/DS Parameters ({name})"].mode()[0]

    final_results.append({
        "Model": name,
        "Accuracy": f"{acc_mean:.4f} ± {acc_std:.4f}",
        "F1 Score": f"{f1_mean:.4f} ± {f1_std:.4f}",
        "Best Params": best_params
    })

final_results_df = pd.DataFrame(final_results)


In [8]:
# SAVE FINAL RESULTS CSV
final_results_df.to_csv("phase2_final_results.csv", index=False)
print("\nFinal aggregated results saved to 'phase2_final_results.csv'")


Final aggregated results saved to 'phase2_final_results.csv'
